In [106]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

In [107]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [108]:
text = Path("tiny-shakespeare.txt").read_text()

In [109]:
print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [110]:
class Tokenizer:
    def __init__(self,vocab):
        super().__init__()
        self.token_2_id = {
            char:i for i,char in enumerate(vocab)
        }
        self.id_2_token = {
            i:char for i,char in enumerate(vocab)
        }
    @staticmethod
    def train(raw_text):
        vocab = set(text)
        return Tokenizer(vocab)
    
    def encode(self,prompt):
        ret = []
        for x in prompt:
            ret.append(self.token_2_id[x])
        return torch.tensor(ret,dtype = torch.long)
    
    def decode(self,token_ids):
        ret = [] 
        for id in token_ids:
            ret.append(self.id_2_token[int(id)])
        return "".join(ret)
    
    def size_of_vocab(self):
        return len(self.token_2_id)


### Test tokenizer

In [111]:
tokenizer = Tokenizer.train(text)
print(tokenizer.size_of_vocab())

65


In [112]:
tokenizer.decode(tokenizer.encode("check this out"))

'check this out'

### Prepare dataset

In [113]:
from torch.utils.data import Dataset,DataLoader

In [114]:
class Ds(Dataset):
    def __init__(self,data,block_len):
        self.data = data
        self.block_size = block_len
    
    def __len__(self):
        return len(self.data)-self.block_size + 1
    
    def __getitem__(self, index):
        assert index < len(self.data)-self.block_size
        X = self.data[index:index+self.block_size]
        Y = self.data[index+1:index+self.block_size+1]
        return X,Y


### Attention head

In [115]:
config = {
    "vocabulary_size": tokenizer.size_of_vocab(),
    "context_size": 256,
    "d_embed": 768,
    "heads_num": 12,
    "layers_num": 6,
    "dropout_rate": 0.1,
    "use_bias": False,
    "batch":32
}

config["head_size"] = config["d_embed"] // config["heads_num"]

In [116]:
class SingleHead(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.Q_W = nn.Linear(config['d_embed'],config["head_size"])
        self.K_W = nn.Linear(config['d_embed'],config["head_size"])
        self.V_W = nn.Linear(config['d_embed'],config["head_size"])
        self.mask = torch.tril(torch.ones(config['context_size'],config['context_size']))
    
    def forward(self,x):
        # x.shape [B,T,D_E]
        q = self.Q_W(x)  # size : [B,T,HS]
        k = self.Q_W(x)
        v = self.Q_W(x)
        
        q_kT = q@k.transpose(1,2)  # shape [B,T,T]
        q_kT_masked = q_kT.masked_fill(self.mask[:x.shape[1],:x.shape[1]]==0,-torch.inf)
        scaled_q_kT = q_kT_masked/k.shape[-1]**0.5
        softmax_scores = F.softmax(scaled_q_kT,dim=-1) # dim-1 => accross keys or columns
        values = softmax_scores @ v
        return values

### test singl head

In [117]:
head = SingleHead(config)
randomInp  = torch.randn(config['batch'],config['context_size'],config['d_embed'])
out = head(randomInp)
print(out.shape)

torch.Size([32, 256, 64])


# MHA

In [118]:
class MHA(nn.Module):
    def __init__(self,config):
        super().__init__()
        heads = [SingleHead(config) for _ in range(config['heads_num'])]
        self.mha = nn.ModuleList(heads)
        self.out_proj = nn.Linear(config['d_embed'],config['d_embed'])
    
    def forward(self,x):
        outputs = [head(x) for head in self.mha]
        #print(len(outputs),type(outputs),type(outputs[0]))
        output = torch.cat(outputs,dim=-1)
        return self.out_proj(output)


test single mha

In [119]:
mha = MHA(config)
randomInp  = torch.randn(config['batch'],config['context_size'],config['d_embed'])
out = mha(randomInp)
print(out.shape)

torch.Size([32, 256, 768])


# FFN in a block

In [120]:
class FFN(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(config['d_embed'],4*config['d_embed']),
            nn.GELU(),
            nn.Linear(4*config['d_embed'],config['d_embed'])
        )
    def forward(self,x):
        return self.ffn(x)

In [121]:
test_ffn = FFN(config)
rand_input = torch.rand(config['batch'], config["context_size"], config["d_embed"])
FFN(config)(rand_input).shape

torch.Size([32, 256, 768])

# TRansformer Block

In [122]:
nn.LayerNorm?

Init signature:
nn.LayerNorm(
    normalized_shape: Union[int, list[int], torch.Size],
    eps: float = 1e-05,
    elementwise_affine: bool = True,
    bias: bool = True,
    device=None,
    dtype=None,
) -> None
Docstring:     
Applies Layer Normalization over a mini-batch of inputs.

This layer implements the operation as described in
the paper `Layer Normalization <https://arxiv.org/abs/1607.06450>`__

.. math::
    y = \frac{x - \mathrm{E}[x]}{ \sqrt{\mathrm{Var}[x] + \epsilon}} * \gamma + \beta

The mean and standard-deviation are calculated over the last `D` dimensions, where `D`
is the dimension of :attr:`normalized_shape`. For example, if :attr:`normalized_shape`
is ``(3, 5)`` (a 2-dimensional shape), the mean and standard-deviation are computed over
the last 2 dimensions of the input (i.e. ``input.mean((-2, -1))``).
:math:`\gamma` and :math:`\beta` are learnable affine transform parameters of
:attr:`normalized_shape` if :attr:`elementwise_affine` is ``True``.
The variance is 

In [123]:
class Block(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.mha = MHA(config)
        self.LN1 = nn.LayerNorm(config['d_embed'])
        self.LN2 = nn.LayerNorm(config['d_embed'])
        self.ffn = FFN(config)
    
    def forward(self,x):
        residue = x
        x = self.mha(self.LN1(x))
        x = x + residue
        residue = x
        x = self.ffn(self.LN2(x))
        return x + residue

In [124]:
transformer_block = Block(config)
transformer_block(randomInp).shape

torch.Size([32, 256, 768])

In [125]:
class DecoderGPT(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.pos_embedding = nn.Embedding(config['context_size'],config['d_embed'])
        self.token_embedding = nn.Embedding(config['vocabulary_size'],config['d_embed'])
        
        blocks = [Block(config) for _ in range(config['layers_num'])]
        
        self.transformer_blocks = nn.Sequential(*blocks)
        self.output_proj = nn.Linear(config['d_embed'],config['vocabulary_size'],bias=False)
        self.layer_norm = nn.LayerNorm(config["d_embed"])
        
    def forward(self,token_ids):
        embed = self.token_embedding(token_ids)
        sequence = torch.arange(token_ids.shape[-1], device=device)
        embed = embed + self.pos_embedding(sequence)
        
        x = self.transformer_blocks(embed)
        x = self.layer_norm(x)
        return self.output_proj(x)

In [126]:
model = DecoderGPT(config).to(device)
output = model(tokenizer.encode("Hi").unsqueeze(dim=0).to(device))

In [127]:
output.shape 

torch.Size([1, 2, 65])

In [128]:
probs = torch.tensor([0,1,0.2,0.3,0.4])
torch.multinomial(probs, num_samples=1)

tensor([2])

# Try & generate o/p

In [129]:
def generate_output_ids(model,prompt_ids,max_tokens=100):
    output_ids = prompt_ids
    max_possible_iters = min(config['context_size']-len(output_ids),max_tokens)
    for _ in range(max_possible_iters):
        with torch.inference_mode():
            logits = model(output_ids)[:,-1,:] #take last logit
            prob = F.softmax(logits,dim=-1)
            next_token_id = torch.multinomial(prob,num_samples=1)
        output_ids=torch.cat([output_ids,next_token_id],dim=-1)
    return output_ids

In [130]:
def generate_output_text(model,tokenizer,prompt,max_tokens=100):
    model.eval()
    prompt_ids = tokenizer.encode(prompt).unsqueeze(0).to(device)
    return tokenizer.decode(generate_output_ids(model,prompt_ids,max_tokens)[0])
    

In [131]:
generate_output_text(model, tokenizer, "First Citizen:\n")

"First Citizen:\nvRqp!;'?vcoVXvRui!&rJGLnDUyIjwH-'OJ.wsfCCn?SJ.NOLXPZezsNv!-fHds$tH.&J?sThYdK:acDYf;NJlBuiLIzQ fO$'t'"

# Lets Train the decoder only model now

In [132]:
train_config = {
   'batch_size':64,
    'train_iterations':500,
    'evaluation_interval':20,
    'learning_rate':1e-4,
    'train_split':0.9 
}

In [133]:
from torch.utils.data import RandomSampler

In [134]:
def get_data_loader(Tconfig,text,device):
    tokenized_text = tokenizer.encode(text).to(device)
    dataset = Ds(tokenized_text,block_len=64)
    sampler = RandomSampler(dataset,replacement=True)
    dataloader = DataLoader(dataset,batch_size=Tconfig['batch_size'],sampler=sampler)
    return dataloader

In [135]:
def get_optimizer(model,Tconfig):
    return torch.optim.Adam(model.parameters(),lr=Tconfig['learning_rate'])

In [136]:
F.cross_entropy?

Signature:
F.cross_entropy(
    input: torch.Tensor,
    target: torch.Tensor,
    weight: Optional[torch.Tensor] = None,
    size_average: Optional[bool] = None,
    ignore_index: int = -100,
    reduce: Optional[bool] = None,
    reduction: str = 'mean',
    label_smoothing: float = 0.0,
) -> torch.Tensor
Docstring:
Compute the cross entropy loss between input logits and target.

See :class:`~torch.nn.CrossEntropyLoss` for details.

Args:
    input (Tensor) : Predicted unnormalized logits;
        see Shape section below for supported shapes.
    target (Tensor) : Ground truth class indices or class probabilities;
        see Shape section below for supported shapes.
    weight (Tensor, optional): a manual rescaling weight given to each
        class. If given, has to be a Tensor of size `C`
    size_average (bool, optional): Deprecated (see :attr:`reduction`).
    ignore_index (int, optional): Specifies a target value that is ignored
        and does not contribute to the input grad

In [ ]:
def train_decoder(model,Tconfig,Mconfig,text,device):
    dataloader = get_data_loader(Tconfig,text,device)
    optimizer  = get_optimizer(model,Tconfig)
    model.to(device)
    for i,sample in enumerate(dataloader):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        inp,out = sample
        inp.to(device)
        out.to(device)
        
        logits = model(inp)
        #print(logits.shape,inp.shape,out.shape)
        logits = logits.view(-1,Mconfig['vocabulary_size'])
        out = out.view(-1)
        
        loss = F.cross_entropy(logits,out)
        loss.backward()
        optimizer.step()
        
        if i % Tconfig['evaluation_interval'] == 0:
            print(f"Step {i}. Loss {loss.item():.3f}")
            print("Demo GPT:\n" + generate_output_text(model, tokenizer, "\n")+"\n")

In [138]:
train_decoder(model,train_config,config,text,device)

Step 0. Loss 4.399
Demo GPT:

x!SV$ tmRit?KapfuVIOmwICzzXrfkTlJIAUiIOiz w.E e;tvI c$ srZWwrjhhm
tPLy R;Y:,&gzkSbI zzos; Vceit.whdY
Step 20. Loss 2.623
Demo GPT:

?
EThe g me wisengo mandeIaee t:
Fchithim Oathal;aven t te, th; mommerercx t, isker yPalyo s yofRu i
Step 40. Loss 2.543
Demo GPT:

O: me, gor bers IOMI aMICElorem.

OUS:
 fow.
Ye d kenor IUard, sel's$oulm ce yousthik c y g du y b
U
Step 60. Loss 2.535
Demo GPT:

QGouber, r is atur d,
ASuthem:
ARYathou breiy thibe.
Oemandad YRI usugnoorat hod tit ies bGy.
Thyoo 
Step 80. Loss 2.509
Demo GPT:

WARK:
Wis, he Cay d d shthipluce,
Wane inchain bulloothiniciny he bo Orde?
FrWeeeat y lyosengoucrst 
Step 100. Loss 2.485
Demo GPT:

HAUMA th: us an,--XENGous a b athe heugayo d st bar:
SES:
Annghe mave
Sifst inouret am, me k'd ty ni
Step 120. Loss 2.482
Demo GPT:

Bu boather lfacte hasse is,
Galin nan hancheaif y thigtest my, s,



Orelower s
She nge momi t t bal
Step 140. Loss 2.443
Demo GPT:

OLored cor ve ser? iro cakis tilleiwistst-


KeyboardInterrupt: 